# TarmacLook 03 — Comparison, statistics, figures, dashboard bundle

**Accelerator: None (CPU).** No GPU quota.

**Inputs to attach:** outputs of notebooks **01**, **02A**, **02B**, and (for the seed replication) **02C**, **02D**.

Produces (in `/kaggle/working`):
- `tarmaclook_paper/tables/`: comparison matrix (CSV + LaTeX), per-class, per-size, per-country, bootstrap CIs, pairwise significance, pilot results
- `tarmaclook_paper/figures/`: PNG (300 dpi) + PDF for every figure
- `tarmaclook_dashboard_bundle/` + `.zip`: weights, metrics, tables, figures, and `registry.json` for the dashboard

Save with **Save Version → Save & Run All**, then on the version page use **Output → New Dataset**. That creates the permanent `tarmaclook-results` Kaggle Dataset.

In [ ]:
%%writefile /kaggle/working/tarmaclook_core.py
# ======================================================================
# TarmacLook core pipeline  (shared by notebooks 01 / 02 / 03)
#
#   01  build_frozen_dataset()   CPU   harmonize RDD2022 + RDD2024, audit,
#                                      dedup, pre-resize, freeze to tar files
#   02  materialize_dataset()    GPU   unpack frozen dataset, train models
#       launch_training()              under an equal wall-time budget,
#       evaluate_model()               evaluate on the untouched test split
#   03  compare + bundle               tables, figures, bootstrap CIs,
#                                      qualitative panels, dashboard bundle
#
# Single source of truth: notebooks embed this file with %%writefile.
# ======================================================================
from __future__ import annotations

import argparse
import contextlib
import datetime as _dt
import hashlib
import io
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
import tarfile
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

CORE_VERSION = "1.1.1"  # 1.1.0: configurable training seed (02C/02D), seed-aware 03; 1.1.1: result scan ignores dashboard-bundle copies

# ----------------------------------------------------------------------
# Project constants
# ----------------------------------------------------------------------
CLASS_NAMES = ["Longitudinal Crack", "Transverse Crack", "Alligator Crack", "Other Corruption", "Pothole"]
NUM_CLASSES = len(CLASS_NAMES)
CRACK_IDS = (0, 1, 2)
POTHOLE_ID = 4
CLASS_COLORS = {0: "#1f77b4", 1: "#ff7f0e", 2: "#2ca02c", 3: "#9467bd", 4: "#d62728"}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLITS = ("train", "val", "test")
DCODE_TO_CLASS = {"D00": 0, "D10": 1, "D20": 2, "D40": 4}
STORE_LONG_SIDE = 640  # images are pre-resized to the network input scale (aspect preserved)

MANIFEST_NAME = "tarmaclook_manifest.csv"
SUMMARY_NAME = "tarmaclook_dataset_summary.json"

MODEL_ZOO = {
    "yolov5nu": "yolov5nu.pt",
    "yolov8n": "yolov8n.pt",
    "yolov9t": "yolov9t.pt",
    "yolov10n": "yolov10n.pt",
    "yolo11n": "yolo11n.pt",
    "yolo12n": "yolo12n.pt",
}
DISPLAY_NAMES = {
    "yolov5nu": "YOLOv5nu", "yolov8n": "YOLOv8n", "yolov9t": "YOLOv9t",
    "yolov10n": "YOLOv10n", "yolo11n": "YOLO11n", "yolo12n": "YOLO12n",
}

# Fixed training protocol: identical for every model. `time` (hours) is passed
# separately; with `time` set, Ultralytics re-estimates the epoch count after
# epoch 1 so the LR schedule and close_mosaic fit inside the budget.
PROTOCOL = dict(
    imgsz=640, batch=16, epochs=300,  # epochs is nominal; the time budget decides
    optimizer="SGD", lr0=0.01, lrf=0.01, momentum=0.937, weight_decay=0.0005,
    warmup_epochs=1.0, close_mosaic=3, cos_lr=False,
    seed=42, deterministic=False, patience=1000,
    workers=2, cache=False, amp=True, save_period=-1, plots=True, val=True,
)
PRIMARY_SEED = PROTOCOL["seed"]  # the main comparison, figures and dashboard use this seed's runs
EVAL_ARGS = dict(imgsz=640, conf=0.001, iou=0.7, max_det=300, batch=16)
DEPLOY_DEFAULTS = dict(conf=0.25, iou=0.7, imgsz=640)


def log(*a):
    print(*a, flush=True)


def now_iso():
    return _dt.datetime.now(_dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def display_name(key):
    return DISPLAY_NAMES.get(key, key)


# ----------------------------------------------------------------------
# Environment report
# ----------------------------------------------------------------------
def env_report():
    info = {"python": sys.version.split()[0], "platform": platform.platform(), "cpu_count": os.cpu_count(),
            "core_version": CORE_VERSION, "time_utc": now_iso()}
    try:
        import torch
        info["torch"] = torch.__version__
        info["cuda"] = torch.cuda.is_available()
        info["gpus"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    except Exception as e:  # pragma: no cover
        info["torch"] = f"unavailable ({e})"
    try:
        import ultralytics
        info["ultralytics"] = ultralytics.__version__
    except Exception:
        info["ultralytics"] = None
    try:
        du = shutil.disk_usage("/kaggle/working" if Path("/kaggle/working").exists() else ".")
        info["working_free_gb"] = round(du.free / 1e9, 1)
        du = shutil.disk_usage("/tmp" if Path("/tmp").exists() else ".")
        info["tmp_free_gb"] = round(du.free / 1e9, 1)
    except Exception:
        pass
    for k, v in info.items():
        log(f"{k:18s}: {v}")
    return info


# ----------------------------------------------------------------------
# Input discovery (does not descend into images/ or labels/ folders)
# ----------------------------------------------------------------------
_SKIP_DESCEND = {"images", "labels"}


def _walk_dirs(root, max_depth=8):
    root = Path(root)
    stack = [(root, 0)]
    while stack:
        d, depth = stack.pop()
        yield d
        if depth >= max_depth or d.name in _SKIP_DESCEND:
            continue
        try:
            subs = sorted((p for p in d.iterdir() if p.is_dir()), reverse=True)
        except (PermissionError, OSError):
            continue
        stack.extend((s, depth + 1) for s in subs)


def find_rdd2022_root(input_root="/kaggle/input"):
    cands = [d for d in _walk_dirs(input_root)
             if all((d / s / "images").is_dir() and (d / s / "labels").is_dir() for s in SPLITS)]
    if not cands:
        raise FileNotFoundError(f"No RDD2022 root with train/val/test images+labels found under {input_root}. "
                                "Attach the RDD2022 dataset as an input.")
    cands.sort(key=lambda p: (p.name != "RDD_SPLIT", len(p.parts)))
    if len(cands) > 1:
        log("WARNING: several train/val/test roots found, using the first:\n  " + "\n  ".join(map(str, cands)))
    return cands[0]


def find_rdd2024_root(input_root="/kaggle/input"):
    cands = [d for d in _walk_dirs(input_root) if d.name == "Training and Validation Dataset"]
    if not cands:
        for d in _walk_dirs(input_root):
            try:
                if any(k.is_dir() and k.name.endswith("_txt") and (k / "data.yaml").exists() for k in d.iterdir()):
                    cands.append(d)
            except OSError:
                pass
    if not cands:
        raise FileNotFoundError(f"No RDD2024 root (country *_txt folders) found under {input_root}.")
    if len(cands) > 1:
        log("WARNING: several RDD2024 roots found, using the first:\n  " + "\n  ".join(map(str, cands)))
    return cands[0]


def find_frozen_dir(*roots):
    roots = roots or ("/kaggle/input", "/kaggle/working")
    hits = []
    for r in roots:
        if not Path(r).exists():
            continue
        for d in _walk_dirs(r, max_depth=6):
            if (d / MANIFEST_NAME).exists() and (d / SUMMARY_NAME).exists():
                hits.append(d)
    if not hits:
        raise FileNotFoundError("Frozen TarmacLook dataset not found. Attach the output of notebook 01 "
                                f"(folder containing {MANIFEST_NAME}) as an input.")
    hits.sort(key=lambda d: (d / SUMMARY_NAME).stat().st_mtime, reverse=True)
    if len(hits) > 1:
        log("WARNING: several frozen datasets found, using the newest:", hits[0])
    return hits[0]


def _names_to_dict(names):
    if isinstance(names, dict):
        return {int(k): str(v) for k, v in names.items()}
    return {i: str(v) for i, v in enumerate(names)}


def read_rdd2022_names(rdd2022_root):
    root = Path(rdd2022_root)
    for d in [root, *list(root.parents)[:3]]:
        if str(d) in ("/", "/kaggle", "/kaggle/input"):
            break
        for y in sorted(d.glob("*.yaml")):
            try:
                cfg = yaml.safe_load(y.read_text(errors="replace"))
            except Exception:
                continue
            if isinstance(cfg, dict) and "names" in cfg:
                return str(y), _names_to_dict(cfg["names"])
    return None, None


def name_to_dcode(name):
    """Map an RDD2024 class name to a D-code. Fixes the D00-only regex in the earlier resume notebook."""
    s = str(name).strip().upper()
    m = re.search(r"\bD\s*-?\s*(\d{2})\b", s)
    if m:
        return f"D{m.group(1)}"
    compact = re.sub(r"[^A-Z]", "", s)
    for key, code in (("LONGITUDINAL", "D00"), ("TRANSVERSE", "D10"), ("ALLIGATOR", "D20"), ("POTHOLE", "D40")):
        if key in compact:
            return code
    return None


def country_from_stem(stem):
    m = re.match(r"^(.*?)_\d+$", stem)
    return m.group(1) if m else "unknown"


def _sanitize(s):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", s)


# ----------------------------------------------------------------------
# Label parsing / validation
# ----------------------------------------------------------------------
def parse_label_lines(lines, idx_map, unmapped_is_error=False, tol=0.02):
    """Validate YOLO xywh lines. Returns kept (str lines), rejected [(line,text,reason)], n_unmapped, n_clipped."""
    kept, rejected, unmapped, clipped = [], [], 0, 0
    for ln, raw in enumerate(lines, 1):
        t = raw.strip()
        if not t:
            continue
        parts = t.split()
        if len(parts) != 5:
            rejected.append((ln, t, "expected_5_fields"))
            continue
        try:
            src = int(float(parts[0]))
            x, y, w, h = (float(v) for v in parts[1:])
        except (ValueError, OverflowError):
            rejected.append((ln, t, "non_numeric"))
            continue
        if src not in idx_map:
            if unmapped_is_error:
                rejected.append((ln, t, f"class_id_out_of_range:{src}"))
            else:
                unmapped += 1
            continue
        if not all(math.isfinite(v) for v in (x, y, w, h)):
            rejected.append((ln, t, "non_finite"))
            continue
        x1, y1, x2, y2 = x - w / 2, y - h / 2, x + w / 2, y + h / 2
        if min(x1, y1) < -tol or max(x2, y2) > 1 + tol:
            rejected.append((ln, t, "coordinate_outside_0_1"))
            continue
        c = (min(max(x1, 0.0), 1.0), min(max(y1, 0.0), 1.0), min(max(x2, 0.0), 1.0), min(max(y2, 0.0), 1.0))
        if c != (x1, y1, x2, y2):
            clipped += 1
        nw, nh = c[2] - c[0], c[3] - c[1]
        if nw <= 1e-6 or nh <= 1e-6:
            rejected.append((ln, t, "non_positive_width_or_height"))
            continue
        kept.append(f"{idx_map[src]} {(c[0] + c[2]) / 2:.6f} {(c[1] + c[3]) / 2:.6f} {nw:.6f} {nh:.6f}")
    return kept, rejected, unmapped, clipped


# ----------------------------------------------------------------------
# 01: build the frozen dataset
# ----------------------------------------------------------------------
def _image_index(img_dir):
    idx = {}
    if img_dir.is_dir():
        for p in img_dir.iterdir():
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                idx.setdefault(p.stem, p)
    return idx


def collect_sources(rdd2022_root, rdd2024_root):
    rdd2022_root, rdd2024_root = Path(rdd2022_root), Path(rdd2024_root)
    rows, maps, orphans = [], {}, []
    for split in SPLITS:
        lbl_dir = rdd2022_root / split / "labels"
        for stem, img in sorted(_image_index(rdd2022_root / split / "images").items()):
            lbl = lbl_dir / f"{stem}.txt"
            rows.append(dict(split=split, source="RDD2022", country=country_from_stem(stem), root_key="rdd2022",
                             src_rel=str(img.relative_to(rdd2022_root)),
                             label_src=str(lbl) if lbl.exists() else "", out_stem=stem, src_split=split,
                             map_key="rdd2022"))
    for cdir in sorted(p for p in rdd2024_root.iterdir() if p.is_dir() and p.name.endswith("_txt")):
        yml = cdir / "data.yaml"
        if not yml.exists():
            log("Skipping RDD2024 folder without data.yaml:", cdir.name)
            continue
        names = _names_to_dict((yaml.safe_load(yml.read_text(errors="replace")) or {}).get("names", []))
        idx_map, table = {}, []
        for i, n in sorted(names.items()):
            code = name_to_dcode(n)
            tgt = DCODE_TO_CLASS.get(code)
            if tgt is not None:
                idx_map[i] = tgt
            table.append(dict(source_idx=i, source_name=n, dcode=code,
                              target=tgt, target_name=CLASS_NAMES[tgt] if tgt is not None else "DROPPED"))
        maps[cdir.name] = dict(idx_map=idx_map, table=table)
        country = cdir.name[:-4]
        if not idx_map:
            log(f"WARNING: {cdir.name}: no source class maps to the 5-class taxonomy -> all its images quarantined")
        for ssplit in ("train", "valid", "val"):
            lbl_dir, img_dir = cdir / ssplit / "labels", cdir / ssplit / "images"
            if not lbl_dir.is_dir():
                continue
            imgs = _image_index(img_dir)
            seen = set()
            for lbl in sorted(lbl_dir.glob("*.txt")):
                img = imgs.get(lbl.stem)
                seen.add(lbl.stem)
                rows.append(dict(split="train", source="RDD2024", country=country, root_key="rdd2024",
                                 src_rel=str(img.relative_to(rdd2024_root)) if img else "",
                                 label_src=str(lbl), out_stem=_sanitize(f"rdd2024_{country}_{ssplit}_{lbl.stem}"),
                                 src_split=ssplit, map_key=cdir.name))
            orphans += [dict(country_folder=cdir.name, split=ssplit, image=imgs[s].name)
                        for s in sorted(set(imgs) - seen)]
    df = pd.DataFrame(rows)
    dup = df.duplicated(["split", "out_stem"], keep="first")
    if dup.any():
        df.loc[dup, "out_stem"] = df.loc[dup, "out_stem"] + "_dup" + df.index[dup].astype(str)
    return df, maps, orphans


_ROOTS = {}
_MAPS = {}


def _init_worker(roots, maps, staging, long_side):
    _ROOTS.update(roots)
    _MAPS.update(maps)
    _ROOTS["_staging"] = staging
    _ROOTS["_long_side"] = long_side


def _process_row(row):
    import cv2
    out = dict(idx=row["idx"], status="ok", n_boxes=0, n_rejected=0, n_unmapped=0, n_clipped=0,
               md5="", stored_md5="", label_sha1="", orig_w=0, orig_h=0, w=0, h=0, out_ext="", rejected=[])
    if not row["src_rel"]:
        out["status"] = "missing_image"
        return out
    if not row["label_src"]:
        out["status"] = "missing_label"
        return out
    lines = Path(row["label_src"]).read_text(errors="replace").splitlines()
    if row["map_key"] == "rdd2022":
        kept, rej, unm, clp = parse_label_lines(lines, {i: i for i in range(NUM_CLASSES)}, unmapped_is_error=True)
    else:
        kept, rej, unm, clp = parse_label_lines(lines, _MAPS[row["map_key"]]["idx_map"])
    out.update(n_boxes=len(kept), n_rejected=len(rej), n_unmapped=unm, n_clipped=clp, rejected=rej)
    if row["source"] == "RDD2024" and not kept:
        out["status"] = "quarantined_no_mapped_boxes"
        return out
    src = Path(_ROOTS[row["root_key"]]) / row["src_rel"]
    data = src.read_bytes()
    out["md5"] = hashlib.md5(data).hexdigest()
    arr = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    if arr is None:
        out["status"] = "corrupt_image"
        return out
    h, w = arr.shape[:2]
    r = _ROOTS["_long_side"] / max(h, w)
    split_dir = Path(_ROOTS["_staging"]) / row["split"]
    if r < 1:
        nw, nh = max(1, round(w * r)), max(1, round(h * r))
        ok, buf = cv2.imencode(".jpg", cv2.resize(arr, (nw, nh), interpolation=cv2.INTER_AREA),
                               [cv2.IMWRITE_JPEG_QUALITY, 95])
        stored, ext = buf.tobytes(), ".jpg"
    else:
        nw, nh, stored, ext = w, h, data, src.suffix.lower()
    (split_dir / "images" / f"{row['out_stem']}{ext}").write_bytes(stored)
    label_text = "\n".join(kept) + ("\n" if kept else "")
    (split_dir / "labels" / f"{row['out_stem']}.txt").write_bytes(label_text.encode())  # exact bytes that are hashed
    out.update(orig_w=w, orig_h=h, w=nw, h=nh, out_ext=ext, stored_md5=hashlib.md5(stored).hexdigest(),
               label_sha1=hashlib.sha1(label_text.encode()).hexdigest())
    return out


def dataset_fingerprint(manifest):
    ok = manifest[manifest.status == "ok"].sort_values(["split", "out_stem"])
    h = hashlib.sha256()
    for s, st, e, ls, im in zip(ok.split, ok.out_stem, ok.out_ext, ok.label_sha1, ok.stored_md5):
        h.update(f"{s}/{st}{e}:{ls}:{im}\n".encode())
    return h.hexdigest()


def build_frozen_dataset(out_dir="/kaggle/working/tarmaclook_frozen", input_root="/kaggle/input",
                         staging="/tmp/tl_staging", long_side=STORE_LONG_SIDE, workers=None,
                         rdd2022_root=None, rdd2024_root=None):
    import multiprocessing as mp
    t0 = time.time()
    out_dir, staging = Path(out_dir), Path(staging)
    rdd2022_root = Path(rdd2022_root) if rdd2022_root else find_rdd2022_root(input_root)
    rdd2024_root = Path(rdd2024_root) if rdd2024_root else find_rdd2024_root(input_root)
    log("RDD2022 root:", rdd2022_root)
    log("RDD2024 root:", rdd2024_root)
    yml_path, rdd2022_names = read_rdd2022_names(rdd2022_root)
    log("RDD2022 data.yaml:", yml_path, "->", rdd2022_names)
    if rdd2022_names and len(rdd2022_names) != NUM_CLASSES:
        raise RuntimeError(f"RDD2022 has {len(rdd2022_names)} classes, expected {NUM_CLASSES}. Check taxonomy.")

    df, maps, orphans = collect_sources(rdd2022_root, rdd2024_root)
    log("\nRDD2024 class mapping per country folder:")
    for k, v in maps.items():
        log(f"  {k}: " + ", ".join(f"{t['source_idx']}:{t['source_name']}->{t['target_name']}" for t in v["table"]))
    log(f"\nCollected {len(df)} candidate images "
        f"({(df.source == 'RDD2022').sum()} RDD2022, {(df.source == 'RDD2024').sum()} RDD2024)")

    for p in (staging, out_dir):
        if p.exists():
            shutil.rmtree(p)
    for s in SPLITS:
        (staging / s / "images").mkdir(parents=True)
        (staging / s / "labels").mkdir(parents=True)
    out_dir.mkdir(parents=True)

    df = df.reset_index(drop=True)
    df["idx"] = df.index
    rows = df[["idx", "split", "source", "root_key", "src_rel", "label_src", "out_stem", "map_key"]].to_dict("records")
    roots = {"rdd2022": str(rdd2022_root), "rdd2024": str(rdd2024_root)}
    workers = workers or max(1, (os.cpu_count() or 2))
    results = []
    ctx = mp.get_context("fork") if hasattr(os, "fork") else mp.get_context()
    with ctx.Pool(workers, initializer=_init_worker, initargs=(roots, maps, str(staging), long_side)) as pool:
        for i, r in enumerate(pool.imap_unordered(_process_row, rows, chunksize=64), 1):
            results.append(r)
            if i % 5000 == 0 or i == len(rows):
                log(f"  processed {i}/{len(rows)}  ({time.time() - t0:.0f}s)")
    res = pd.DataFrame(results).set_index("idx").sort_index()
    rej_rows = []
    for idx, rej in res["rejected"].items():
        for ln, text, reason in rej:
            rej_rows.append(dict(split=df.at[idx, "split"], source=df.at[idx, "source"],
                                 out_stem=df.at[idx, "out_stem"], label_src=df.at[idx, "label_src"],
                                 line=ln, content=text, reason=reason))
    man = df.drop(columns=["idx"]).join(res.drop(columns=["rejected"]))

    # exact-duplicate removal (md5 of original bytes)
    dup_rows = []
    ok = man[man.status == "ok"]
    for md5, g in ok.groupby("md5"):
        if len(g) < 2:
            continue
        eval_members = g[g.split != "train"]
        train_members = g[g.split == "train"]
        keep_first_train = eval_members.empty
        for j, (i, r) in enumerate(train_members.iterrows()):
            if keep_first_train and j == 0:
                continue
            man.at[i, "status"] = "dup_of_eval" if not eval_members.empty else "dup_in_train"
        dup_rows += [dict(md5=md5, split=r.split, out_stem=r.out_stem, source=r.source,
                          status_after=man.at[i, "status"]) for i, r in g.iterrows()]
    for i, r in man[man.status.isin(["dup_of_eval", "dup_in_train"])].iterrows():
        for p in (staging / r.split / "images" / f"{r.out_stem}{r.out_ext}", staging / r.split / "labels" / f"{r.out_stem}.txt"):
            p.unlink(missing_ok=True)

    man["fingerprint_member"] = man.status == "ok"
    fp = dataset_fingerprint(man)

    # tar per split (uncompressed: images are already JPEG)
    for s in SPLITS:
        with tarfile.open(out_dir / f"tarmaclook_{s}.tar", "w") as t:
            t.add(staging / s, arcname=s)
        log(f"  wrote tarmaclook_{s}.tar ({(out_dir / f'tarmaclook_{s}.tar').stat().st_size / 1e9:.2f} GB)")
    shutil.rmtree(staging, ignore_errors=True)

    man.to_csv(out_dir / MANIFEST_NAME, index=False)
    pd.DataFrame(rej_rows).to_csv(out_dir / "annotation_rejections.csv", index=False)
    pd.DataFrame(dup_rows).to_csv(out_dir / "exact_duplicates.csv", index=False)
    pd.DataFrame(orphans).to_csv(out_dir / "rdd2024_images_without_labels.csv", index=False)
    pd.DataFrame([dict(country_folder=k, **t) for k, v in maps.items() for t in v["table"]]).to_csv(
        out_dir / "rdd2024_class_mapping.csv", index=False)

    summary = dataset_summary(man, out_dir)
    summary.update(
        fingerprint=fp, core_version=CORE_VERSION, created_utc=now_iso(), store_long_side=long_side,
        class_names=CLASS_NAMES, rdd2022_root=str(rdd2022_root), rdd2024_root=str(rdd2024_root),
        rdd2022_yaml=yml_path, rdd2022_names=rdd2022_names, dcode_to_class=DCODE_TO_CLASS,
        rejected_annotations=len(rej_rows),
        rejection_reasons=dict(Counter(r["reason"] for r in rej_rows)),
        clipped_boxes=int(man.n_clipped.sum()), rdd2024_unmapped_boxes=int(man[man.source == "RDD2024"].n_unmapped.sum()),
        rdd2024_images_without_labels=len(orphans), build_seconds=round(time.time() - t0),
    )
    (out_dir / SUMMARY_NAME).write_text(json.dumps(summary, indent=2, default=str))
    log(f"\nFrozen dataset written to {out_dir}\nfingerprint: {fp}\nbuild time: {summary['build_seconds']} s")
    return man, summary


def _label_class_counts(staging_or_dataset, man):
    counts = defaultdict(Counter)
    for s in SPLITS:
        sub = man[(man.split == s) & (man.status == "ok")]
        for stem in sub.out_stem:
            p = Path(staging_or_dataset) / s / "labels" / f"{stem}.txt"
            if p.exists():
                for line in p.read_text().splitlines():
                    if line.strip():
                        counts[s][int(line.split()[0])] += 1
    return counts


def dataset_summary(man, out_dir=None):
    ok = man[man.status == "ok"]
    s = {
        "status_counts": man.status.value_counts().to_dict(),
        "images_per_split": ok.split.value_counts().reindex(SPLITS, fill_value=0).to_dict(),
        "images_per_split_source": {f"{a}|{b}": int(n) for (a, b), n in ok.groupby(["split", "source"]).size().items()},
        "boxes_per_split": ok.groupby("split").n_boxes.sum().reindex(SPLITS, fill_value=0).astype(int).to_dict(),
        "empty_label_images_per_split": ok[ok.n_boxes == 0].split.value_counts().reindex(SPLITS, fill_value=0).to_dict(),
        "empty_label_images_by_country": {f"{a}|{b}": int(n) for (a, b), n in
                                          ok[ok.n_boxes == 0].groupby(["split", "country"]).size().items()},
        "images_per_country_split": {f"{a}|{b}": int(n) for (a, b), n in ok.groupby(["split", "country"]).size().items()},
        "quarantined_by_country": man[man.status == "quarantined_no_mapped_boxes"].country.value_counts().to_dict(),
    }
    return s


def dataset_class_counts_from_tar(frozen_dir):
    """Per-split class box counts read straight from the frozen tar files (no extraction)."""
    counts = {}
    for s in SPLITS:
        c = Counter()
        with tarfile.open(Path(frozen_dir) / f"tarmaclook_{s}.tar") as t:
            for m in t:
                if m.isfile() and "/labels/" in m.name and m.name.endswith(".txt"):
                    for line in t.extractfile(m).read().decode().splitlines():
                        if line.strip():
                            c[int(line.split()[0])] += 1
        counts[s] = {CLASS_NAMES[k]: int(c.get(k, 0)) for k in range(NUM_CLASSES)}
    return counts


def plot_dataset_figures(frozen_dir, fig_dir=None):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    frozen_dir = Path(frozen_dir)
    fig_dir = Path(fig_dir or frozen_dir / "figures")
    fig_dir.mkdir(parents=True, exist_ok=True)
    man = pd.read_csv(frozen_dir / MANIFEST_NAME)
    ok = man[man.status == "ok"]
    counts = dataset_class_counts_from_tar(frozen_dir)
    cdf = pd.DataFrame(counts)
    cdf.to_csv(fig_dir.parent / "class_box_counts.csv")
    ax = cdf.plot(kind="bar", figsize=(8, 4.2), color=["#4c72b0", "#dd8452", "#55a868"])
    ax.set_ylabel("Bounding boxes")
    ax.set_title("Class distribution per split (frozen TarmacLook dataset)")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    for ext in ("png", "pdf"):
        plt.savefig(fig_dir / f"dataset_class_distribution.{ext}", dpi=300)
    plt.close()
    pc = ok.groupby(["country", "split"]).size().unstack(fill_value=0).reindex(columns=list(SPLITS), fill_value=0)
    pc.to_csv(fig_dir.parent / "images_per_country_split.csv")
    ax = pc.plot(kind="barh", stacked=True, figsize=(8, max(3, 0.35 * len(pc))))
    ax.set_xlabel("Images")
    ax.set_title("Images per country / source and split")
    plt.tight_layout()
    for ext in ("png", "pdf"):
        plt.savefig(fig_dir / f"dataset_country_split.{ext}", dpi=300)
    plt.close()
    log("Dataset figures saved to", fig_dir)
    return cdf, pc


# ----------------------------------------------------------------------
# 02: materialize, train, evaluate
# ----------------------------------------------------------------------
def materialize_dataset(frozen_dir, dest="/tmp/tarmaclook_dataset", verify=True):
    frozen_dir, dest = Path(frozen_dir), Path(dest)
    summary = json.loads((frozen_dir / SUMMARY_NAME).read_text())
    man = pd.read_csv(frozen_dir / MANIFEST_NAME, keep_default_na=False)
    marker = dest / ".tarmaclook_fingerprint"
    if marker.exists() and marker.read_text().strip() == summary["fingerprint"]:
        log("Dataset already materialized at", dest)
    else:
        if dest.exists():
            shutil.rmtree(dest)
        dest.mkdir(parents=True)
        for s in SPLITS:
            t0 = time.time()
            with tarfile.open(frozen_dir / f"tarmaclook_{s}.tar") as t:
                try:
                    t.extractall(dest, filter="data")
                except TypeError:
                    t.extractall(dest)
            log(f"  extracted {s} in {time.time() - t0:.0f}s")
        if verify:
            ok = man[man.status == "ok"].copy()
            label_sha, img_md5, missing = [], [], 0
            for s, st, e in zip(ok.split, ok.out_stem, ok.out_ext):
                lp, ip = dest / s / "labels" / f"{st}.txt", dest / s / "images" / f"{st}{e}"
                if not (lp.exists() and ip.exists()):
                    missing += 1
                    label_sha.append("")
                    img_md5.append("")
                    continue
                label_sha.append(hashlib.sha1(lp.read_bytes()).hexdigest())
                img_md5.append(hashlib.md5(ip.read_bytes()).hexdigest())
            ok["label_sha1"], ok["stored_md5"] = label_sha, img_md5
            fp = dataset_fingerprint(ok)
            if missing or fp != summary["fingerprint"]:
                raise RuntimeError(f"Frozen dataset verification FAILED (missing={missing}, fingerprint {fp} != "
                                   f"{summary['fingerprint']}). Do not train on this copy.")
            log("Fingerprint verified:", fp)
        marker.write_text(summary["fingerprint"])
    data_yaml = dest / "data.yaml"
    data_yaml.write_text(yaml.safe_dump({
        "path": str(dest), "train": "train/images", "val": "val/images", "test": "test/images",
        "nc": NUM_CLASSES, "names": {i: n for i, n in enumerate(CLASS_NAMES)},
    }, sort_keys=False))
    return data_yaml, man, summary


def train_one(model_key, data_yaml, project, time_hours, weights_dir=None, device=0, extra=None, seed=None):
    """Train one model under the fixed protocol (called inside a subprocess, one GPU each).
    `seed` overrides PROTOCOL['seed'] for seed-replication runs; everything else stays fixed."""
    import torch
    import ultralytics
    from ultralytics import YOLO
    weights = MODEL_ZOO.get(model_key, model_key)
    if weights_dir and (Path(weights_dir) / weights).exists():
        weights = str(Path(weights_dir) / weights)
    args = dict(PROTOCOL)
    args.update(extra or {})
    if seed is not None:
        args["seed"] = int(seed)
    run_dir = Path(project) / model_key
    prov = dict(model_key=model_key, weights=weights, protocol=args, time_hours=time_hours,
                data=str(data_yaml), ultralytics=ultralytics.__version__, torch=torch.__version__,
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
                cuda_visible_devices=os.environ.get("CUDA_VISIBLE_DEVICES"), start_utc=now_iso(),
                core_version=CORE_VERSION)
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "tarmaclook_provenance.json").write_text(json.dumps(prov, indent=2))
    t0 = time.time()
    YOLO(weights).train(data=str(data_yaml), project=str(project), name=model_key, exist_ok=True,
                        device=device, time=time_hours, **args)
    prov.update(end_utc=now_iso(), wall_hours=round((time.time() - t0) / 3600, 3))
    (run_dir / "tarmaclook_provenance.json").write_text(json.dumps(prov, indent=2))
    return run_dir


def _results_progress(run_dir):
    p = Path(run_dir) / "results.csv"
    if not p.exists():
        return "starting"
    try:
        d = pd.read_csv(p)
        d.columns = [c.strip() for c in d.columns]
        last = d.iloc[-1]
        return (f"epoch {int(last['epoch'])}  val mAP50={last.get('metrics/mAP50(B)', float('nan')):.4f}  "
                f"mAP50-95={last.get('metrics/mAP50-95(B)', float('nan')):.4f}")
    except Exception as e:
        return f"results.csv unreadable ({e})"


def launch_training(model_keys, data_yaml, project, time_hours, weights_dir, log_dir,
                    core_path=None, poll_s=60, report_every_s=900, device="0", seed=None):
    """Train models in parallel, one per GPU (queued if fewer GPUs). Returns {model_key: exit_code}."""
    import torch
    core_path = core_path or str(Path(__file__).resolve())
    n_gpu = max(1, torch.cuda.device_count()) if device != "cpu" else 1
    project, log_dir = Path(project), Path(log_dir)
    project.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)
    queue, running, codes = list(model_keys), {}, {}
    free = list(range(n_gpu))
    last_report = 0
    t_start = time.time()
    while queue or running:
        while queue and free:
            key, gpu = queue.pop(0), free.pop(0)
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
            cmd = [sys.executable, core_path, "train", "--model", key, "--data", str(data_yaml),
                   "--project", str(project), "--time", str(time_hours), "--weights-dir", str(weights_dir),
                   "--device", device]
            if seed is not None:
                cmd += ["--seed", str(int(seed))]
            fh = open(log_dir / f"train_{key}.log", "w")
            running[key] = (subprocess.Popen(cmd, env=env, stdout=fh, stderr=subprocess.STDOUT,
                                             cwd=str(weights_dir)), gpu, fh)
            log(f"[{now_iso()}] started {key} on GPU {gpu} (log: {log_dir / f'train_{key}.log'})")
        time.sleep(poll_s)
        for key in list(running):
            proc, gpu, fh = running[key]
            if proc.poll() is not None:
                fh.close()
                codes[key] = proc.returncode
                free.append(gpu)
                del running[key]
                log(f"[{now_iso()}] {key} finished with exit code {proc.returncode}: {_results_progress(project / key)}")
                if proc.returncode != 0:
                    log(f"---- last lines of train_{key}.log ----")
                    log("\n".join((log_dir / f"train_{key}.log").read_text(errors="replace").splitlines()[-40:]))
        if time.time() - last_report > report_every_s and running:
            last_report = time.time()
            log(f"[{now_iso()}] elapsed {(time.time() - t_start) / 3600:.2f} h")
            for key in running:
                log(f"   {key:10s} {_results_progress(project / key)}")
    return codes


def load_split_gt(dataset_dir, man, split="test"):
    """Ground truth in stored-image pixel coords. Returns dict of arrays."""
    sub = man[(man.split == split) & (man.status == "ok")].sort_values("out_stem").reset_index(drop=True)
    img_idx, cls, boxes = [], [], []
    for i, r in sub.iterrows():
        p = Path(dataset_dir) / split / "labels" / f"{r.out_stem}.txt"
        for line in p.read_text().splitlines():
            if not line.strip():
                continue
            c, x, y, w, h = line.split()
            x, y, w, h = float(x) * r.w, float(y) * r.h, float(w) * r.w, float(h) * r.h
            img_idx.append(i)
            cls.append(int(c))
            boxes.append((x - w / 2, y - h / 2, x + w / 2, y + h / 2))
    return dict(stems=sub.out_stem.tolist(), exts=sub.out_ext.tolist(), countries=sub.country.tolist(),
                widths=sub.w.to_numpy(), heights=sub.h.to_numpy(),
                img_idx=np.asarray(img_idx, np.int32), cls=np.asarray(cls, np.int16),
                xyxy=np.asarray(boxes, np.float32).reshape(-1, 4))


def predict_split(model, dataset_dir, gt, split="test", device=0):
    stem_to_idx = {s: i for i, s in enumerate(gt["stems"])}
    paths = [str(Path(dataset_dir) / split / "images" / f"{s}{e}") for s, e in zip(gt["stems"], gt["exts"])]
    I, C, S, B = [], [], [], []
    for k in range(0, len(paths), 256):
        for r in model.predict(source=paths[k:k + 256], stream=True, device=device, verbose=False,
                               imgsz=EVAL_ARGS["imgsz"], conf=EVAL_ARGS["conf"], iou=EVAL_ARGS["iou"],
                               max_det=EVAL_ARGS["max_det"]):
            b = r.boxes
            n = len(b)
            if n:
                I.append(np.full(n, stem_to_idx[Path(r.path).stem], np.int32))
                C.append(b.cls.cpu().numpy().astype(np.int16))
                S.append(b.conf.cpu().numpy().astype(np.float32))
                B.append(b.xyxy.cpu().numpy().astype(np.float32))
    cat = lambda L, dt, shape=None: (np.concatenate(L) if L else np.zeros(shape or 0, dt))
    return dict(img_idx=cat(I, np.int32), cls=cat(C, np.int16), score=cat(S, np.float32),
                xyxy=cat(B, np.float32, (0, 4)).reshape(-1, 4))


def save_predictions(path, gt, pred):
    np.savez_compressed(path, stems=np.asarray(gt["stems"]), **pred)


def load_predictions(path):
    z = np.load(path, allow_pickle=False)
    return {k: z[k] for k in ("img_idx", "cls", "score", "xyxy")}, z["stems"].tolist()


def _coco_objects(gt, pred):
    from pycocotools.coco import COCO
    images = [dict(id=i, width=int(w), height=int(h), file_name=s)
              for i, (s, w, h) in enumerate(zip(gt["stems"], gt["widths"], gt["heights"]))]
    anns = []
    for k, (i, c, b) in enumerate(zip(gt["img_idx"], gt["cls"], gt["xyxy"])):
        w, h = float(b[2] - b[0]), float(b[3] - b[1])
        anns.append(dict(id=k + 1, image_id=int(i), category_id=int(c) + 1, bbox=[float(b[0]), float(b[1]), w, h],
                         area=w * h, iscrowd=0))
    cg = COCO()
    cg.dataset = dict(info={}, licenses=[], images=images, annotations=anns,
                      categories=[dict(id=i + 1, name=n) for i, n in enumerate(CLASS_NAMES)])
    with contextlib.redirect_stdout(io.StringIO()):
        cg.createIndex()
    if len(pred["score"]):
        x = pred["xyxy"]
        arr = np.column_stack([pred["img_idx"], x[:, 0], x[:, 1], x[:, 2] - x[:, 0], x[:, 3] - x[:, 1],
                               pred["score"], pred["cls"].astype(np.int64) + 1]).astype(np.float64)
        with contextlib.redirect_stdout(io.StringIO()):
            cd = cg.loadRes(arr)
    else:
        cd = None
    return cg, cd


def _coco_run(cg, cd, img_ids=None):
    from pycocotools.cocoeval import COCOeval
    E = COCOeval(cg, cd, "bbox")  # standard COCO maxDets [1, 10, 100]
    if img_ids is not None:
        E.params.imgIds = list(img_ids)
    with contextlib.redirect_stdout(io.StringIO()):
        E.evaluate()
        E.accumulate()
        E.summarize()
    return E


def _prec_mean(p):
    p = p[p > -1]
    return float(np.mean(p)) if p.size else float("nan")


def coco_breakdown(gt, pred):
    """COCO-style AP by class, object size (at stored/network scale) and country."""
    cg, cd = _coco_objects(gt, pred)
    if cd is None:
        return {}, pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    E = _coco_run(cg, cd)
    P = E.eval["precision"]  # [T, R, K, A, M]
    st = E.stats
    overall = dict(coco_AP=st[0], coco_AP50=st[1], coco_AP75=st[2], coco_AP_small=st[3], coco_AP_medium=st[4],
                   coco_AP_large=st[5], coco_AR100=st[8], coco_AR_small=st[9], coco_AR_medium=st[10], coco_AR_large=st[11])
    overall = {k: (float(v) if v > -1 else float("nan")) for k, v in overall.items()}  # -1 = no GT in bucket
    gt_count = np.bincount(gt["cls"], minlength=NUM_CLASSES)
    areas = (gt["xyxy"][:, 2] - gt["xyxy"][:, 0]) * (gt["xyxy"][:, 3] - gt["xyxy"][:, 1])
    size_bin = np.digitize(areas, [32 ** 2, 96 ** 2])
    per_class, per_size = [], []
    for k in range(NUM_CLASSES):
        per_class.append(dict(class_id=k, class_name=CLASS_NAMES[k], gt_boxes=int(gt_count[k]),
                              coco_AP50=_prec_mean(P[0, :, k, 0, -1]), coco_AP=_prec_mean(P[:, :, k, 0, -1])))
        for a, name in ((1, "small"), (2, "medium"), (3, "large")):
            per_size.append(dict(class_id=k, class_name=CLASS_NAMES[k], size=name,
                                 gt_boxes=int(((gt["cls"] == k) & (size_bin == a - 1)).sum()),
                                 coco_AP50=_prec_mean(P[0, :, k, a, -1]), coco_AP=_prec_mean(P[:, :, k, a, -1])))
    for a, name in ((1, "small"), (2, "medium"), (3, "large")):
        per_size.append(dict(class_id=-1, class_name="ALL", size=name, gt_boxes=int((size_bin == a - 1).sum()),
                             coco_AP50=_prec_mean(P[0, :, :, a, -1]), coco_AP=_prec_mean(P[:, :, :, a, -1])))
    per_country = []
    countries = np.asarray(gt["countries"])
    for c in sorted(set(countries)):
        ids = np.where(countries == c)[0]
        Ec = _coco_run(cg, cd, ids.tolist())
        per_country.append(dict(country=c, images=len(ids), gt_boxes=int(np.isin(gt["img_idx"], ids).sum()),
                                coco_AP50=float(Ec.stats[1]) if Ec.stats[1] > -1 else float("nan"),
                                coco_AP=float(Ec.stats[0]) if Ec.stats[0] > -1 else float("nan")))
    return overall, pd.DataFrame(per_class), pd.DataFrame(per_size), pd.DataFrame(per_country)


def model_complexity(yolo_model, imgsz=640):
    params = int(sum(p.numel() for p in yolo_model.model.parameters()))
    gflops = float("nan")
    try:
        from ultralytics.utils.torch_utils import get_flops
        gflops = float(get_flops(yolo_model.model, imgsz=imgsz))
    except Exception:
        try:
            gflops = float(yolo_model.info(verbose=False)[3])
        except Exception:
            pass
    return params, gflops


def latency_benchmark(weights, image_paths, device, n=200, warmup=15):
    import cv2
    import torch
    from ultralytics import YOLO
    m = YOLO(str(weights))
    imgs = [cv2.imread(p) for p in image_paths[:n]]
    kw = dict(imgsz=DEPLOY_DEFAULTS["imgsz"], conf=DEPLOY_DEFAULTS["conf"], iou=DEPLOY_DEFAULTS["iou"],
              device=device, verbose=False)
    for im in imgs[:warmup]:
        m.predict(im, **kw)
    cuda = str(device) != "cpu" and torch.cuda.is_available()
    if cuda:
        torch.cuda.synchronize()
    sp = []
    t0 = time.perf_counter()
    for im in imgs:
        sp.append(m.predict(im, **kw)[0].speed)
    if cuda:
        torch.cuda.synchronize()
    el = time.perf_counter() - t0
    return dict(device=(torch.cuda.get_device_name(0) if cuda else f"cpu x{os.cpu_count()}"), images=len(imgs),
                fps=len(imgs) / el, e2e_ms=1000 * el / len(imgs),
                preprocess_ms=float(np.mean([s["preprocess"] for s in sp])),
                inference_ms=float(np.mean([s["inference"] for s in sp])),
                postprocess_ms=float(np.mean([s["postprocess"] for s in sp])))


def training_summary(run_dir):
    run_dir = Path(run_dir)
    out = {}
    p = run_dir / "results.csv"
    if p.exists():
        d = pd.read_csv(p)
        d.columns = [c.strip() for c in d.columns]
        col = "metrics/mAP50-95(B)"
        b = int(d[col].idxmax()) if col in d else len(d) - 1
        out.update(epochs_completed=int(len(d)), best_epoch=int(d.loc[b, "epoch"]),
                   val_mAP50_best=float(d.loc[b].get("metrics/mAP50(B)", np.nan)),
                   val_mAP50_95_best=float(d.loc[b].get(col, np.nan)),
                   train_hours=float(d["time"].iloc[-1] / 3600) if "time" in d else None)
    prov = run_dir / "tarmaclook_provenance.json"
    if prov.exists():
        out["provenance"] = json.loads(prov.read_text())
    return out


def evaluate_model(model_key, run_dir, data_yaml, dataset_dir, man, out_root, device=0,
                   n_latency_gpu=200, n_latency_cpu=50):
    """Test-split evaluation: Ultralytics metrics + COCO breakdowns + complexity + latency. Saves a results folder."""
    import torch
    import ultralytics
    from ultralytics import YOLO
    run_dir, out = Path(run_dir), Path(out_root) / model_key
    out.mkdir(parents=True, exist_ok=True)
    best = run_dir / "weights" / "best.pt"
    if not best.exists():
        raise FileNotFoundError(f"{best} missing - training did not produce a checkpoint")
    model = YOLO(str(best))

    log(f"[{model_key}] Ultralytics test-split validation")
    m = model.val(data=str(data_yaml), split="test", device=device, plots=True, project=str(out),
                  name="ultralytics_test", exist_ok=True, verbose=True,
                  imgsz=EVAL_ARGS["imgsz"], conf=EVAL_ARGS["conf"], iou=EVAL_ARGS["iou"],
                  max_det=EVAL_ARGS["max_det"], batch=EVAL_ARGS["batch"])
    box = m.box
    per_class = []
    for i, c in enumerate(np.asarray(box.ap_class_index, int)):
        per_class.append(dict(class_id=int(c), class_name=CLASS_NAMES[int(c)], precision=float(box.p[i]),
                              recall=float(box.r[i]), f1=float(box.f1[i]), AP50=float(box.ap50[i]),
                              AP50_95=float(box.ap[i])))
    pc = pd.DataFrame(per_class)

    def _cls(cid, col):
        v = pc.loc[pc.class_id == cid, col]
        return float(v.iloc[0]) if len(v) else float("nan")

    P, R = float(box.mp), float(box.mr)
    headline = dict(
        mAP50=float(box.map50), mAP50_95=float(box.map), mAP75=float(box.map75), precision=P, recall=R,
        f1=(2 * P * R / (P + R)) if (P + R) else 0.0,
        crack_AP50=float(np.nanmean([_cls(c, "AP50") for c in CRACK_IDS])),
        crack_AP50_95=float(np.nanmean([_cls(c, "AP50_95") for c in CRACK_IDS])),
        pothole_AP50=_cls(POTHOLE_ID, "AP50"), pothole_AP50_95=_cls(POTHOLE_ID, "AP50_95"),
        val_speed_ms=dict(m.speed),
    )

    log(f"[{model_key}] raw test predictions + COCO breakdowns")
    gt = load_split_gt(dataset_dir, man, "test")
    pred = predict_split(model, dataset_dir, gt, "test", device)
    save_predictions(out / "predictions_test.npz", gt, pred)
    coco_overall, coco_cls, coco_size, coco_country = coco_breakdown(gt, pred)

    params, gflops = model_complexity(model)
    paths = [str(Path(dataset_dir) / "test" / "images" / f"{s}{e}") for s, e in zip(gt["stems"], gt["exts"])]
    rng = np.random.default_rng(0)
    sample = [paths[i] for i in rng.permutation(len(paths))[:max(n_latency_gpu, n_latency_cpu)]]
    log(f"[{model_key}] latency benchmark")
    lat_gpu = latency_benchmark(best, sample, device, n=n_latency_gpu) if torch.cuda.is_available() else None
    lat_cpu = latency_benchmark(best, sample, "cpu", n=n_latency_cpu, warmup=5)

    train_info = training_summary(run_dir)
    train_protocol = (train_info.get("provenance") or {}).get("protocol") or dict(PROTOCOL)
    metrics = dict(
        model_key=model_key, model=display_name(model_key), seed=int(train_protocol.get("seed", PRIMARY_SEED)),
        created_utc=now_iso(), core_version=CORE_VERSION,
        ultralytics=ultralytics.__version__, torch=torch.__version__, split="test",
        test_images=len(gt["stems"]), test_boxes=int(len(gt["cls"])),
        **headline, **coco_overall, parameters=params, gflops=gflops,
        weights_mb=round(best.stat().st_size / 1e6, 2), latency_gpu=lat_gpu, latency_cpu=lat_cpu,
        training=train_info, protocol=train_protocol, eval_args=EVAL_ARGS, deploy_defaults=DEPLOY_DEFAULTS,
        class_names=CLASS_NAMES,
    )
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2, default=float))
    pc.to_csv(out / "per_class.csv", index=False)
    coco_cls.to_csv(out / "per_class_coco.csv", index=False)
    coco_size.to_csv(out / "per_size.csv", index=False)
    coco_country.to_csv(out / "per_country.csv", index=False)
    (out / "weights").mkdir(exist_ok=True)
    shutil.copy2(best, out / "weights" / "best.pt")
    art = out / "train_artifacts"
    art.mkdir(exist_ok=True)
    for f in run_dir.iterdir():
        if f.is_file() and f.suffix.lower() in {".csv", ".yaml", ".json", ".png", ".jpg"}:
            shutil.copy2(f, art / f.name)
    log(f"[{model_key} seed {metrics['seed']}] test mAP50={headline['mAP50']:.4f} mAP50-95={headline['mAP50_95']:.4f} "
        f"pothole AP50={headline['pothole_AP50']:.4f} crack AP50={headline['crack_AP50']:.4f} "
        f"params={params / 1e6:.2f}M GFLOPs={gflops:.1f}")
    return metrics


# ----------------------------------------------------------------------
# 03: comparison, statistics, figures, dashboard bundle
# ----------------------------------------------------------------------
def result_seed(m):
    """Training seed of a metrics.json dict. Results from core 1.0.0 have no top-level 'seed';
    their seed is read from the training provenance (falls back to PRIMARY_SEED)."""
    if m.get("seed") is not None:
        return int(m["seed"])
    prov = ((m.get("training") or {}).get("provenance") or {}).get("protocol") or {}
    return int(prov.get("seed", (m.get("protocol") or {}).get("seed", PRIMARY_SEED)))


def _scan_results(roots):
    roots = roots or ("/kaggle/input", "/kaggle/working")
    found = {}  # (model_key, seed) -> (dir, metrics)
    for r in roots:
        if not Path(r).exists():
            continue
        for d in _walk_dirs(r, max_depth=7):
            mj = d / "metrics.json"
            if not mj.is_file():
                continue
            try:
                m = json.loads(mj.read_text())
            except Exception:
                continue
            if "model_key" not in m or not (mj.parent / "predictions_test.npz").exists():
                continue
            if "tarmaclook_dashboard_bundle" in d.parts:  # bundle copies lack train_artifacts/; use the 02 outputs
                continue
            k = (m["model_key"], result_seed(m))
            if k not in found or m.get("created_utc", "") > found[k][1].get("created_utc", ""):
                found[k] = (mj.parent, m)
    return found


def _model_order(k):
    order = list(MODEL_ZOO)
    return order.index(k) if k in order else 99


def find_result_dirs(*roots, seed=PRIMARY_SEED):
    """{model_key: result_dir} for one training seed (default: the primary seed 42)."""
    found = {mk: v[0] for (mk, s), v in _scan_results(roots).items() if s == seed}
    return {k: found[k] for k in sorted(found, key=_model_order)}


def find_all_result_dirs(*roots):
    """{(model_key, seed): result_dir} for every seed found."""
    found = _scan_results(roots)
    return {k: found[k][0] for k in sorted(found, key=lambda ks: (_model_order(ks[0]), ks[1]))}


def seed_replication(all_dirs):
    """Per-run table and per-model summary across training seeds.
    Returns (runs, summary, rank_table). With 2 seeds, 'sd' is the sample SD of 2 values (= |diff| / sqrt 2)."""
    metrics = ["mAP50", "mAP50_95", "precision", "recall", "f1", "crack_AP50", "pothole_AP50"]
    rows = []
    for (k, s), d in all_dirs.items():
        m = json.loads((Path(d) / "metrics.json").read_text())
        t = m.get("training", {})
        rows.append(dict(key=k, Model=display_name(k), seed=s, **{c: m.get(c) for c in metrics},
                         epochs=t.get("epochs_completed"), best_epoch=t.get("best_epoch"),
                         train_hours=t.get("train_hours"),
                         gpu_latency_ms=(m.get("latency_gpu") or {}).get("e2e_ms")))
    runs = pd.DataFrame(rows)
    if runs.empty:
        return runs, runs, runs
    g = runs.groupby(["key", "Model"], sort=False)
    summary = g.agg(n_seeds=("seed", "nunique"), seeds=("seed", lambda x: ",".join(map(str, sorted(x)))),
                    **{f"{c}_mean": (c, "mean") for c in metrics},
                    **{f"{c}_sd": (c, "std") for c in metrics},
                    **{f"{c}_range": (c, lambda x: float(x.max() - x.min())) for c in ("mAP50", "mAP50_95")},
                    epochs_mean=("epochs", "mean")).reset_index()
    summary = summary.iloc[sorted(range(len(summary)), key=lambda i: _model_order(summary.key.iloc[i]))]
    ranks = runs.pivot(index="Model", columns="seed", values="mAP50_95").rank(ascending=False, method="min")
    ranks.columns = [f"rank mAP50-95 (seed {c})" for c in ranks.columns]
    return runs, summary.reset_index(drop=True), ranks.reset_index()


def plot_seed_replication(runs, fig_dir):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    models = list(dict.fromkeys(runs.sort_values("key", key=lambda s: s.map(_model_order)).Model))
    for ax, col, lab in zip(axes, ["mAP50", "mAP50_95"], ["mAP@50", "mAP@50:95"]):
        for i, mdl in enumerate(models):
            v = runs[runs.Model == mdl]
            ax.scatter(np.full(len(v), i), v[col], s=36, zorder=3, label=None)
            for _, r in v.iterrows():
                ax.annotate(f"s{int(r.seed)}", (i, r[col]), xytext=(6, -3), textcoords="offset points", fontsize=7)
            ax.hlines(v[col].mean(), i - 0.25, i + 0.25, color="k", lw=1.5, zorder=2)
        ax.set_xticks(range(len(models)))
        ax.set_xticklabels(models)
        ax.set_ylabel(f"test {lab}")
        ax.grid(axis="y", alpha=0.3)
    fig.suptitle("Seed replication (points = training seeds, bar = mean)", fontsize=10)
    _save(fig, fig_dir, "fig_seed_replication")
    plt.close(fig)


def comparison_table(result_dirs):
    rows = []
    for k, d in result_dirs.items():
        m = json.loads((Path(d) / "metrics.json").read_text())
        t = m.get("training", {})
        rows.append({
            "Model": m["model"], "key": k,
            "mAP@50": m["mAP50"], "mAP@50:95": m["mAP50_95"], "Precision": m["precision"], "Recall": m["recall"],
            "F1": m["f1"], "Crack AP@50": m["crack_AP50"], "Pothole AP@50": m["pothole_AP50"],
            "Pothole AP@50:95": m["pothole_AP50_95"],
            "AP small": m.get("coco_AP_small"), "AP medium": m.get("coco_AP_medium"), "AP large": m.get("coco_AP_large"),
            "Params (M)": m["parameters"] / 1e6, "GFLOPs": m["gflops"], "Weights (MB)": m["weights_mb"],
            "GPU latency (ms)": (m.get("latency_gpu") or {}).get("e2e_ms"),
            "GPU FPS": (m.get("latency_gpu") or {}).get("fps"),
            "CPU latency (ms)": (m.get("latency_cpu") or {}).get("e2e_ms"),
            "Epochs": t.get("epochs_completed"), "Best epoch": t.get("best_epoch"),
            "Train hours": t.get("train_hours"),
        })
    return pd.DataFrame(rows)


def _iou_matrix(a, b):
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / np.maximum(aa[:, None] + ab[None, :] - inter, 1e-9)


def match_iou50(gt, pred, n_images, max_det_per_image=100, iou_thr=0.5):
    """COCO-style greedy matching at IoU 0.5 per class. Returns per-class dict with sorted det arrays."""
    def _topk_mask(img, score, k):  # top-k per image, like COCO maxDets (applied per class)
        if not k or not len(img):
            return np.ones(len(img), bool)
        order = np.lexsort((-score, img))
        rank = np.zeros(len(order), int)
        s_img = img[order]
        starts = np.r_[0, np.flatnonzero(np.diff(s_img)) + 1]
        for s, e in zip(starts, np.r_[starts[1:], len(order)]):
            rank[order[s:e]] = np.arange(e - s)
        return rank < k

    out = {}
    for c in range(NUM_CLASSES):
        gm = gt["cls"] == c
        g_img, g_box = gt["img_idx"][gm], gt["xyxy"][gm]
        pm = pred["cls"] == c
        p_img, p_sc, p_box = pred["img_idx"][pm], pred["score"][pm], pred["xyxy"][pm]
        km = _topk_mask(p_img, p_sc, max_det_per_image)
        p_img, p_sc, p_box = p_img[km], p_sc[km], p_box[km]
        tp = np.zeros(len(p_img), bool)
        g_by = defaultdict(list)
        for j, i in enumerate(g_img):
            g_by[int(i)].append(j)
        p_by = defaultdict(list)
        for j, i in enumerate(p_img):
            p_by[int(i)].append(j)
        for i, pj in p_by.items():
            gj = g_by.get(i)
            if not gj:
                continue
            pj = np.asarray(pj)[np.argsort(-p_sc[pj], kind="stable")]
            ious = _iou_matrix(p_box[pj], g_box[gj])
            used = np.zeros(len(gj), bool)
            for r, j in enumerate(pj):
                cand = np.where(~used, ious[r], -1)
                b = int(np.argmax(cand))
                if cand[b] >= iou_thr:
                    used[b] = True
                    tp[j] = True
        o = np.argsort(-p_sc, kind="stable")
        out[c] = dict(img=p_img[o], tp=tp[o], ngt=np.bincount(g_img, minlength=n_images))
    return out


_RS = np.linspace(0, 1, 101)


def _ap101_weighted(m, w):
    npos = float((m["ngt"] * w).sum())
    if npos <= 0:
        return float("nan")
    if len(m["tp"]) == 0:
        return 0.0
    wd = w[m["img"]].astype(np.float64)
    tpw = np.cumsum(wd * m["tp"])
    fpw = np.cumsum(wd * ~m["tp"])
    rec = tpw / npos
    prec = tpw / np.maximum(tpw + fpw, 1e-12)
    prec = np.maximum.accumulate(prec[::-1])[::-1]
    idx = np.searchsorted(rec, _RS, side="left")
    return float(np.where(idx < len(prec), prec[np.minimum(idx, len(prec) - 1)], 0.0).mean())


def bootstrap_map50(gt, preds_by_model, n_boot=1000, seed=0):
    """Paired image-level bootstrap of COCO-style mAP@50 (and per-class AP@50) for several models."""
    n = len(gt["stems"])
    matched = {k: match_iou50(gt, p, n) for k, p in preds_by_model.items()}
    ones = np.ones(n)
    point = {k: {c: _ap101_weighted(mm[c], ones) for c in range(NUM_CLASSES)} for k, mm in matched.items()}
    rng = np.random.default_rng(seed)
    boots = {k: np.zeros((n_boot, NUM_CLASSES)) for k in matched}
    for b in range(n_boot):
        w = np.bincount(rng.integers(0, n, n), minlength=n).astype(np.float64)
        for k, mm in matched.items():
            boots[k][b] = [_ap101_weighted(mm[c], w) for c in range(NUM_CLASSES)]
    rows = []
    for k in matched:
        mp = np.nanmean(boots[k], axis=1)
        rows.append(dict(key=k, Model=display_name(k), mAP50=float(np.nanmean(list(point[k].values()))),
                         ci_low=float(np.percentile(mp, 2.5)), ci_high=float(np.percentile(mp, 97.5)),
                         **{f"AP50 {CLASS_NAMES[c]}": point[k][c] for c in range(NUM_CLASSES)},
                         **{f"CI {CLASS_NAMES[c]}": f"[{np.nanpercentile(boots[k][:, c], 2.5):.3f}, "
                                                    f"{np.nanpercentile(boots[k][:, c], 97.5):.3f}]"
                            for c in range(NUM_CLASSES)}))
    pair_rows = []
    keys = list(matched)
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            a, b = keys[i], keys[j]
            d = np.nanmean(boots[a], 1) - np.nanmean(boots[b], 1)
            p = 2 * min((d <= 0).mean(), (d >= 0).mean())
            pair_rows.append(dict(model_a=display_name(a), model_b=display_name(b),
                                  diff_mAP50=float(np.nanmean(list(point[a].values())) - np.nanmean(list(point[b].values()))),
                                  ci_low=float(np.percentile(d, 2.5)), ci_high=float(np.percentile(d, 97.5)),
                                  p_bootstrap=float(min(1.0, p)), significant_0_05=bool(p < 0.05)))
    return pd.DataFrame(rows), pd.DataFrame(pair_rows)


def _save(fig, fig_dir, name):
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(Path(fig_dir) / f"{name}.{ext}", dpi=300, bbox_inches="tight")


def make_figures(result_dirs, comp, fig_dir):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig_dir = Path(fig_dir)
    fig_dir.mkdir(parents=True, exist_ok=True)
    names = comp["Model"].tolist()
    x = np.arange(len(names))

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(x - 0.2, comp["mAP@50"], 0.4, label="mAP@50")
    ax.bar(x + 0.2, comp["mAP@50:95"], 0.4, label="mAP@50:95")
    for i, (a, b) in enumerate(zip(comp["mAP@50"], comp["mAP@50:95"])):
        ax.text(i - 0.2, a, f"{a:.3f}", ha="center", va="bottom", fontsize=8)
        ax.text(i + 0.2, b, f"{b:.3f}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x, names)
    ax.set_ylabel("Test-split score")
    ax.set_title("Detection accuracy on the held-out RDD2022 test split")
    ax.legend()
    _save(fig, fig_dir, "fig_map_comparison")
    plt.close(fig)

    pcs = []
    for k, d in result_dirs.items():
        p = pd.read_csv(Path(d) / "per_class.csv")
        p["Model"] = display_name(k)
        pcs.append(p)
    pc = pd.concat(pcs)
    heat = pc.pivot(index="Model", columns="class_name", values="AP50").reindex(index=names, columns=CLASS_NAMES)
    fig, ax = plt.subplots(figsize=(8, 0.6 * len(names) + 1.5))
    im = ax.imshow(heat.values, cmap="viridis", aspect="auto")
    ax.set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=20, ha="right")
    ax.set_yticks(range(len(names)), names)
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            v = heat.values[i, j]
            ax.text(j, i, "-" if np.isnan(v) else f"{v:.3f}", ha="center", va="center",
                    color="white" if (not np.isnan(v) and v < np.nanmax(heat.values) * 0.6) else "black", fontsize=9)
    fig.colorbar(im, ax=ax, label="AP@50")
    ax.set_title("Per-class AP@50 (test split)")
    _save(fig, fig_dir, "fig_per_class_ap50")
    plt.close(fig)

    for xcol, fname, xl in (("Params (M)", "fig_pareto_params", "Parameters (M)"),
                            ("GFLOPs", "fig_pareto_gflops", "GFLOPs @640"),
                            ("GPU latency (ms)", "fig_pareto_gpu_latency", "End-to-end GPU latency, batch 1 (ms)"),
                            ("CPU latency (ms)", "fig_pareto_cpu_latency", "End-to-end CPU latency, batch 1 (ms)")):
        if comp[xcol].isna().all():
            continue
        fig, ax = plt.subplots(figsize=(5.5, 4))
        ax.scatter(comp[xcol], comp["mAP@50:95"], s=60)
        for _, r in comp.iterrows():
            ax.annotate(r["Model"], (r[xcol], r["mAP@50:95"]), textcoords="offset points", xytext=(5, 4), fontsize=9)
        ax.set_xlabel(xl)
        ax.set_ylabel("mAP@50:95 (test)")
        ax.grid(alpha=0.3)
        ax.set_title("Accuracy vs cost")
        _save(fig, fig_dir, fname)
        plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for k, d in result_dirs.items():
        p = Path(d) / "train_artifacts" / "results.csv"
        if not p.exists():
            continue
        r = pd.read_csv(p)
        r.columns = [c.strip() for c in r.columns]
        axes[0].plot(r["epoch"], r["metrics/mAP50-95(B)"], marker="o", ms=3, label=display_name(k))
        if "time" in r:
            axes[1].plot(r["time"] / 3600, r["metrics/mAP50-95(B)"], marker="o", ms=3, label=display_name(k))
    axes[0].set_xlabel("Epoch")
    axes[1].set_xlabel("Training wall time (h)")
    for a in axes:
        a.set_ylabel("Validation mAP@50:95")
        a.grid(alpha=0.3)
        a.legend()
    fig.suptitle("Training curves under an equal wall-time budget")
    _save(fig, fig_dir, "fig_training_curves")
    plt.close(fig)

    sz = []
    for k, d in result_dirs.items():
        s = pd.read_csv(Path(d) / "per_size.csv")
        s = s[s.class_id == -1].assign(Model=display_name(k))
        sz.append(s)
    if sz:
        s = pd.concat(sz).pivot(index="Model", columns="size", values="coco_AP").reindex(
            index=names, columns=["small", "medium", "large"])
        ax = s.plot(kind="bar", figsize=(7, 4))
        ax.set_ylabel("COCO AP@50:95")
        ax.set_title("AP by object size (COCO thresholds at 640-px scale)")
        plt.xticks(rotation=0)
        _save(ax.figure, fig_dir, "fig_ap_by_size")
        plt.close(ax.figure)

    cc = []
    for k, d in result_dirs.items():
        c = pd.read_csv(Path(d) / "per_country.csv").assign(Model=display_name(k))
        cc.append(c)
    if cc:
        c = pd.concat(cc).pivot(index="country", columns="Model", values="coco_AP50").reindex(columns=names)
        ax = c.plot(kind="bar", figsize=(9, 4))
        ax.set_ylabel("COCO AP@50")
        ax.set_title("Cross-country performance on the test split")
        plt.xticks(rotation=20, ha="right")
        _save(ax.figure, fig_dir, "fig_ap50_by_country")
        plt.close(ax.figure)
    log("Figures saved to", fig_dir)


def qualitative_grid(gt, preds_by_model, dataset_dir, fig_dir, conf=0.25, seed=7, per_class=1):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    import cv2
    rng = np.random.default_rng(seed)
    chosen = []
    for c in range(NUM_CLASSES):
        ids = np.unique(gt["img_idx"][gt["cls"] == c])
        ids = [i for i in ids if i not in chosen]
        if ids:
            chosen += list(rng.choice(ids, size=min(per_class, len(ids)), replace=False))
    cols = 1 + len(preds_by_model)
    fig, axes = plt.subplots(len(chosen), cols, figsize=(3.2 * cols, 3.0 * len(chosen)))
    axes = np.atleast_2d(axes)
    for r, i in enumerate(chosen):
        img = cv2.cvtColor(cv2.imread(str(Path(dataset_dir) / "test" / "images" / f"{gt['stems'][i]}{gt['exts'][i]}")),
                           cv2.COLOR_BGR2RGB)
        panels = [("Ground truth", gt["xyxy"][gt["img_idx"] == i], gt["cls"][gt["img_idx"] == i], None)]
        for k, p in preds_by_model.items():
            m = (p["img_idx"] == i) & (p["score"] >= conf)
            panels.append((display_name(k), p["xyxy"][m], p["cls"][m], p["score"][m]))
        for cidx, (title, boxes, clss, scores) in enumerate(panels):
            ax = axes[r, cidx]
            ax.imshow(img)
            for j, (b, c) in enumerate(zip(boxes, clss)):
                col = CLASS_COLORS[int(c)]
                ax.add_patch(Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, lw=1.6, ec=col))
                if scores is not None:
                    ax.text(b[0], b[1], f"{scores[j]:.2f}", color="white", fontsize=6,
                            bbox=dict(facecolor=col, alpha=0.7, pad=0.5, lw=0))
            ax.set_axis_off()
            if r == 0:
                ax.set_title(title, fontsize=10)
        axes[r, 0].text(-0.02, 0.5, gt["stems"][i], transform=axes[r, 0].transAxes, rotation=90,
                        ha="right", va="center", fontsize=7)
    handles = [Rectangle((0, 0), 1, 1, fc="none", ec=CLASS_COLORS[c], lw=2) for c in range(NUM_CLASSES)]
    fig.legend(handles, CLASS_NAMES, loc="lower center", ncol=NUM_CLASSES, fontsize=9, frameon=False)
    fig.suptitle(f"Qualitative comparison (predictions at conf >= {conf})", y=1.0)
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    for ext in ("png", "pdf"):
        fig.savefig(Path(fig_dir) / f"fig_qualitative.{ext}", dpi=200, bbox_inches="tight")
    plt.close(fig)
    return [gt["stems"][i] for i in chosen]


def write_latex(df, path, caption, label, float_fmt="%.3f"):
    try:
        tex = df.to_latex(index=False, float_format=lambda v: float_fmt % v, caption=caption, label=label,
                          escape=True)
    except Exception:
        tex = df.to_string(index=False)
    Path(path).write_text(tex)


def build_dashboard_bundle(result_dirs, comp, tables_dir, fig_dir, frozen_dir, bundle_dir, extra_info=None):
    bundle = Path(bundle_dir)
    if bundle.exists():
        shutil.rmtree(bundle)
    (bundle / "models").mkdir(parents=True)
    models = []
    for k, d in result_dirs.items():
        d = Path(d)
        dst = bundle / "models" / k
        dst.mkdir()
        for f in ("metrics.json", "per_class.csv", "per_class_coco.csv", "per_size.csv", "per_country.csv",
                  "predictions_test.npz"):
            if (d / f).exists():
                shutil.copy2(d / f, dst / f)
        shutil.copy2(d / "weights" / "best.pt", dst / "best.pt")
        if (d / "train_artifacts" / "results.csv").exists():
            shutil.copy2(d / "train_artifacts" / "results.csv", dst / "results.csv")
        m = json.loads((d / "metrics.json").read_text())
        models.append(dict(key=k, display=m["model"], weights=f"models/{k}/best.pt", metrics=f"models/{k}/metrics.json",
                           mAP50=m["mAP50"], mAP50_95=m["mAP50_95"], pothole_AP50=m["pothole_AP50"],
                           crack_AP50=m["crack_AP50"], params_M=m["parameters"] / 1e6, gflops=m["gflops"],
                           gpu_latency_ms=(m.get("latency_gpu") or {}).get("e2e_ms"),
                           cpu_latency_ms=(m.get("latency_cpu") or {}).get("e2e_ms")))
    shutil.copytree(tables_dir, bundle / "tables")
    shutil.copytree(fig_dir, bundle / "figures")
    (bundle / "dataset").mkdir()
    for f in (SUMMARY_NAME, "class_box_counts.csv", "images_per_country_split.csv", "rdd2024_class_mapping.csv"):
        if (Path(frozen_dir) / f).exists():
            shutil.copy2(Path(frozen_dir) / f, bundle / "dataset" / f)
    man = pd.read_csv(Path(frozen_dir) / MANIFEST_NAME, keep_default_na=False)
    man[(man.split == "test") & (man.status == "ok")][["out_stem", "out_ext", "country", "w", "h", "n_boxes"]].to_csv(
        bundle / "dataset" / "test_images.csv", index=False)
    best = max(models, key=lambda r: r["mAP50_95"]) if models else None
    fastest = min((r for r in models if r["gpu_latency_ms"]), key=lambda r: r["gpu_latency_ms"], default=None)
    registry = dict(
        project="TarmacLook", schema_version=1, created_utc=now_iso(), core_version=CORE_VERSION,
        class_names=CLASS_NAMES, class_colors={CLASS_NAMES[i]: c for i, c in CLASS_COLORS.items()},
        inference_defaults=DEPLOY_DEFAULTS, protocol=PROTOCOL, eval_args=EVAL_ARGS,
        dataset=json.loads((Path(frozen_dir) / SUMMARY_NAME).read_text()),
        models=models, recommended_model=best["key"] if best else None,
        fastest_model=fastest["key"] if fastest else None,
        tables=sorted(p.name for p in (bundle / "tables").iterdir()),
        figures=sorted(p.name for p in (bundle / "figures").iterdir() if p.suffix == ".png"),
        **(extra_info or {}),
    )
    (bundle / "registry.json").write_text(json.dumps(registry, indent=2, default=str))
    zip_path = shutil.make_archive(str(bundle), "zip", root_dir=bundle)
    log("Dashboard bundle:", bundle, "\nZip:", zip_path)
    return bundle, zip_path


# ----------------------------------------------------------------------
# CLI (used by launch_training subprocesses)
# ----------------------------------------------------------------------
def _cli():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    t = sub.add_parser("train")
    t.add_argument("--model", required=True)
    t.add_argument("--data", required=True)
    t.add_argument("--project", required=True)
    t.add_argument("--time", type=float, required=True)
    t.add_argument("--weights-dir", default=None)
    t.add_argument("--device", default="0")
    t.add_argument("--seed", type=int, default=None)
    a = ap.parse_args()
    if a.cmd == "train":
        dev = a.device if a.device == "cpu" else int(a.device)
        train_one(a.model, a.data, a.project, a.time, a.weights_dir, device=dev, seed=a.seed)


if __name__ == "__main__":
    _cli()

In [ ]:
import subprocess, sys
try:
    import pycocotools  # noqa
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocotools"], check=True)

In [ ]:
import sys, importlib, json
sys.path.insert(0, "/kaggle/working")
import tarmaclook_core as tl
importlib.reload(tl)
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40)
ENV = tl.env_report()

In [ ]:
import tarfile
from pathlib import Path
FROZEN = tl.find_frozen_dir("/kaggle/input")
RESULT_DIRS = tl.find_result_dirs("/kaggle/input")            # primary seed (42): main comparison
ALL_RUNS = tl.find_all_result_dirs("/kaggle/input")            # every (model, seed) found
print("models found (seed %d):" % tl.PRIMARY_SEED, list(RESULT_DIRS))
print("all runs found:", list(ALL_RUNS))
EXPECTED = ["yolov5nu", "yolov8n", "yolo11n", "yolo12n"]
missing = [m for m in EXPECTED if m not in RESULT_DIRS]
if missing:
    print("WARNING - missing results for", missing, "(attach the 02A/02B outputs); continuing with what exists")
EVAL_DIR = Path("/tmp/tl_eval")
if not (EVAL_DIR / "test").exists():
    with tarfile.open(FROZEN / "tarmaclook_test.tar") as t:
        try: t.extractall(EVAL_DIR, filter="data")
        except TypeError: t.extractall(EVAL_DIR)
MAN = pd.read_csv(FROZEN / tl.MANIFEST_NAME, keep_default_na=False)
GT = tl.load_split_gt(EVAL_DIR, MAN, "test")
print(len(GT["stems"]), "test images,", len(GT["cls"]), "boxes")
OUT = Path("/kaggle/working/tarmaclook_paper"); TABLES, FIGS = OUT / "tables", OUT / "figures"
TABLES.mkdir(parents=True, exist_ok=True); FIGS.mkdir(parents=True, exist_ok=True)

## Main comparison matrix

In [ ]:
COMP = tl.comparison_table(RESULT_DIRS)
COMP.to_csv(TABLES / "comparison_matrix.csv", index=False)
main_cols = ["Model", "mAP@50", "mAP@50:95", "Precision", "Recall", "F1", "Crack AP@50", "Pothole AP@50",
             "Params (M)", "GFLOPs", "GPU latency (ms)", "CPU latency (ms)", "Epochs", "Train hours"]
tl.write_latex(COMP[main_cols], TABLES / "comparison_matrix.tex",
               "Detection accuracy and cost on the held-out RDD2022 test split (equal wall-time training budget).",
               "tab:comparison")
display(COMP.set_index("Model").T)

In [ ]:
def stack(fname, **kw):
    parts = []
    for k, d in RESULT_DIRS.items():
        p = pd.read_csv(Path(d) / fname); p.insert(0, "Model", tl.display_name(k)); parts.append(p)
    return pd.concat(parts, ignore_index=True)

PER_CLASS = stack("per_class.csv"); PER_CLASS.to_csv(TABLES / "per_class_ultralytics.csv", index=False)
pc_wide = PER_CLASS.pivot(index="Model", columns="class_name", values="AP50").reindex(columns=tl.CLASS_NAMES)
tl.write_latex(pc_wide.reset_index(), TABLES / "per_class_ap50.tex", "Per-class AP@50 on the test split.", "tab:perclass")
PER_SIZE = stack("per_size.csv"); PER_SIZE.to_csv(TABLES / "per_size_coco.csv", index=False)
PER_COUNTRY = stack("per_country.csv"); PER_COUNTRY.to_csv(TABLES / "per_country_coco.csv", index=False)
pcn = PER_COUNTRY.pivot(index="country", columns="Model", values="coco_AP50")
tl.write_latex(pcn.reset_index(), TABLES / "per_country_ap50.tex", "COCO AP@50 per country (test split).", "tab:country")
display(pc_wide); display(PER_SIZE[PER_SIZE.class_id == -1].pivot(index="Model", columns="size", values="coco_AP")); display(pcn)

## Figures

In [ ]:
tl.make_figures(RESULT_DIRS, COMP, FIGS)
from IPython.display import Image as _I
for f in ["fig_map_comparison", "fig_per_class_ap50", "fig_pareto_gpu_latency", "fig_training_curves", "fig_ap_by_size", "fig_ap50_by_country"]:
    if (FIGS / f"{f}.png").exists(): display(_I(filename=str(FIGS / f"{f}.png"), width=700))

## Statistical uncertainty: paired image-level bootstrap (1,000 resamples) of COCO-style mAP@50

In [ ]:
PREDS = {}
for k, d in RESULT_DIRS.items():
    p, stems = tl.load_predictions(Path(d) / "predictions_test.npz")
    assert stems == GT["stems"], f"{k}: test image order mismatch"
    PREDS[k] = p
CI, PAIRS = tl.bootstrap_map50(GT, PREDS, n_boot=1000, seed=0)
CI.to_csv(TABLES / "bootstrap_ci.csv", index=False); PAIRS.to_csv(TABLES / "pairwise_bootstrap.csv", index=False)
tl.write_latex(PAIRS, TABLES / "pairwise_bootstrap.tex",
               "Paired bootstrap differences in mAP@50 (95% CI, two-sided bootstrap p).", "tab:pairs")
display(CI[["Model", "mAP50", "ci_low", "ci_high"]]); display(PAIRS)

## Seed replication (02C/02D): does the ranking survive a different training seed?
The bootstrap above covers test-sample uncertainty only. This section adds training-run variance: every model trained again with seed 2 under the identical 6 h budget.
Skipped automatically if only one seed is attached.

In [ ]:
SEEDS = sorted({s for _, s in ALL_RUNS})
if len(SEEDS) < 2:
    print("Only seed(s)", SEEDS, "attached - attach the 02C/02D outputs for the seed replication. Skipping.")
else:
    RUNS_T, SEED_SUM, RANKS = tl.seed_replication(ALL_RUNS)
    RUNS_T.to_csv(TABLES / "seed_runs.csv", index=False)
    SEED_SUM.to_csv(TABLES / "seed_summary.csv", index=False)
    RANKS.to_csv(TABLES / "seed_ranks.csv", index=False)
    sum_cols = ["Model", "seeds", "mAP50_mean", "mAP50_sd", "mAP50_range", "mAP50_95_mean", "mAP50_95_sd",
                "mAP50_95_range", "pothole_AP50_mean", "epochs_mean"]
    tl.write_latex(SEED_SUM[sum_cols], TABLES / "seed_summary.tex",
                   "Test-split accuracy across training seeds (mean, SD and range over seeds; identical 6 h budget).",
                   "tab:seeds", float_fmt="%.4f")
    tl.plot_seed_replication(RUNS_T, FIGS)
    display(RUNS_T); display(SEED_SUM[sum_cols]); display(RANKS)
    # Paired bootstrap within every non-primary seed (same test images, same procedure as above)
    for s in SEEDS:
        if s == tl.PRIMARY_SEED:
            continue
        P_S = {}
        for (k, ss), d in ALL_RUNS.items():
            if ss != s:
                continue
            p, stems = tl.load_predictions(Path(d) / "predictions_test.npz")
            assert stems == GT["stems"], f"{k} seed {s}: test image order mismatch"
            P_S[k] = p
        CI_S, PAIRS_S = tl.bootstrap_map50(GT, P_S, n_boot=1000, seed=0)
        CI_S.to_csv(TABLES / f"bootstrap_ci_seed{s}.csv", index=False)
        PAIRS_S.to_csv(TABLES / f"pairwise_bootstrap_seed{s}.csv", index=False)
        tl.write_latex(PAIRS_S, TABLES / f"pairwise_bootstrap_seed{s}.tex",
                       f"Paired bootstrap differences in mAP@50, training seed {s}.", f"tab:pairs_seed{s}")
        print(f"seed {s}:"); display(CI_S[["Model", "mAP50", "ci_low", "ci_high"]]); display(PAIRS_S)
    display(_I(filename=str(FIGS / "fig_seed_replication.png"), width=800))

## Qualitative comparison (same test images, all models)

In [ ]:
chosen = tl.qualitative_grid(GT, PREDS, EVAL_DIR, FIGS, conf=0.25, seed=7)
print("images:", chosen)
display(_I(filename=str(FIGS / "fig_qualitative.png"), width=900))

## Pilot experiments (NOT comparable: different data, splits and budgets)
These are earlier runs from the knowledge base. Report them only as preliminary or pilot results, e.g. in an appendix.

In [ ]:
PILOT = pd.DataFrame([
    dict(Experiment="HOG + One-vs-Rest SVM (image classification)", Data="RDD2022 (earlier pipeline)", Metric="Macro-F1", Value=0.4667, Note="classification metric, not detection mAP"),
    dict(Experiment="Faster R-CNN ResNet18-FPN", Data="4,000/800 subset, 5 epochs", Metric="val mAP@50", Value=0.1807, Note="validation subset"),
    dict(Experiment="Faster R-CNN ResNet50-FPN", Data="4,000/800 subset, 5 epochs", Metric="val mAP@50", Value=0.3999, Note="validation subset"),
    dict(Experiment="Faster R-CNN MobileNetV3-L-FPN", Data="4,000/800 subset, 5 epochs", Metric="val mAP@50", Value=0.2901, Note="validation subset"),
    dict(Experiment="Faster R-CNN MobileNetV3-L-320-FPN", Data="4,000/800 subset, 5 epochs", Metric="val mAP@50", Value=0.1568, Note="validation subset"),
    dict(Experiment="YOLOv8n (Colab, 9 of 20 epochs)", Data="old Colab V_2.0.2 merge", Metric="val mAP@50", Value=0.4796, Note="old merge incl. fake background images"),
    # Fill in when the YOLO11m pilot finishes (old Colab merge, not comparable):
    # dict(Experiment="YOLO11m (pilot)", Data="old Colab V_2.0.2 merge", Metric="val mAP@50", Value=None, Note="different data"),
])
PILOT.to_csv(TABLES / "pilot_results.csv", index=False)
display(PILOT)

## Dashboard bundle (weights + metrics + figures + registry.json)

In [ ]:
import shutil
for f in ["class_box_counts.csv", "images_per_country_split.csv"]:
    if (FROZEN / f).exists(): shutil.copy2(FROZEN / f, TABLES / f"dataset_{f}")
if (FROZEN / "figures").exists():
    for f in (FROZEN / "figures").iterdir(): shutil.copy2(f, FIGS / f.name)
(TABLES / "protocol.json").write_text(json.dumps(dict(protocol=tl.PROTOCOL, eval_args=tl.EVAL_ARGS,
    deploy_defaults=tl.DEPLOY_DEFAULTS, dataset_fingerprint=json.loads((FROZEN / tl.SUMMARY_NAME).read_text())["fingerprint"]), indent=2))
BUNDLE, ZIP = tl.build_dashboard_bundle(RESULT_DIRS, COMP, TABLES, FIGS, FROZEN, "/kaggle/working/tarmaclook_dashboard_bundle")
print(json.dumps({k: v for k, v in json.loads((BUNDLE / "registry.json").read_text()).items() if k in ("models", "recommended_model", "fastest_model")}, indent=1, default=str))

## Make it permanent
1. **Save Version → Save & Run All (Commit)**.
2. On the finished version: **Output → New Dataset** → name it `tarmaclook-results`.
3. Download `tarmaclook_dashboard_bundle.zip` for the local Streamlit dashboard. `registry.json` lists every model, its weights path and its headline metrics.